# KCS early-20 official portal PDF transport repair
Source-only, one exact GET.


In [1]:
import datetime,hashlib,json,os,shlex,subprocess,sys,tarfile,tempfile
from pathlib import Path
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,_settings,_ssh
card=root/'research/kcs_early20_official_mirror_pdf_20261023';raw=root/'data/snapshots/kcs_early20_official_mirror_pdf_20261023';raw.mkdir(parents=True,exist_ok=True)
plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
parent=root/plan['parent_capture_card']
for name,k in [('plan.json','parent_plan_sha256'),('result.json','parent_result_sha256'),('summary.json','parent_summary_sha256')]:assert sha(parent/name)==plan[k]
previous=json.loads((parent/'response_receipts.json').read_text());assert previous[2]['source_name']=='government_portal_mirror' and previous[2]['http_status']=='200'
state=card/'run_state.json';state.write_text(json.dumps({'status':'started','plan_sha256':sha(card/'plan.json'),'GETs_started':1},indent=2)+'\n')
body=raw/'attachment.body';headers=raw/'attachment.headers';start=datetime.datetime.now(datetime.timezone.utc).isoformat()
proc=subprocess.run(['curl','--location','--silent','--show-error','--connect-timeout','5','--max-time','20','--dump-header',str(headers),'--output',str(body),'--write-out','%{http_code}|%{url_effective}',plan['source_url']],capture_output=True,text=True)
end=datetime.datetime.now(datetime.timezone.utc).isoformat();parts=proc.stdout.strip().split('|',1)
receipt={'source_url':plan['source_url'],'started_at_utc':start,'completed_at_utc':end,'curl_exit':proc.returncode,'http_status':parts[0] if parts else None,'effective_url':parts[1] if len(parts)>1 else None,'error':proc.stderr[:300] if proc.returncode else None,'body_path':str(body.relative_to(root)),'body_bytes':body.stat().st_size if body.exists() else 0,'body_sha256':sha(body) if body.exists() else None,'headers_path':str(headers.relative_to(root)),'headers_bytes':headers.stat().st_size if headers.exists() else 0,'headers_sha256':sha(headers) if headers.exists() else None}
(card/'response_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
valid=receipt['http_status']=='200' and body.exists() and body.read_bytes().startswith(b'%PDF-');info=None
if valid:
 proc_info=subprocess.run(['pdfinfo',str(body)],capture_output=True,text=True,timeout=20)
 if proc_info.returncode==0:
  info={'pages':next((int(line.split(':',1)[1].strip()) for line in proc_info.stdout.splitlines() if line.startswith('Pages:')),None),'pdfinfo_exit':0}
 else:valid=False;info={'pdfinfo_exit':proc_info.returncode,'error':proc_info.stderr[:300]}
result={'status':'PDF_bytes_valid_visual_semantics_pending' if valid else 'needs_source_transport_or_PDF_validation','receipt':receipt,'valid_PDF':valid,'pdfinfo':info,'semiconductor_table_semantics_qualified':False,'historical_first_public_bytes_qualified':False,'strategy_forward_day_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
entries={str(p.relative_to(root)):sha(p) for p in sorted(raw.iterdir()) if p.is_file()};entries[str((card/'plan.json').relative_to(root))]=sha(card/'plan.json');entries[str((card/'response_receipt.json').relative_to(root))]=sha(card/'response_receipt.json')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='kcs-pdf-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for rel in sorted(entries):out.add(root/rel,arcname=rel,recursive=False)
 digest=sha(archive);size=archive.stat().st_size
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert evidence['sha256']==digest and evidence['size_bytes']==size
 nas_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(nas_path)).split()[0];assert remote==digest
 with tempfile.TemporaryDirectory(prefix='kcs-pdf-readback-') as back:
  fetched=Path(back)/'restore.tar.gz'
  with fetched.open('wb') as output:
   proc2=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'cat '+shlex.quote(nas_path)],stdout=output,stderr=subprocess.PIPE,timeout=90)
  assert proc2.returncode==0 and sha(fetched)==digest
  with tarfile.open(fetched,'r:gz') as source:
   members={m.name:m for m in source.getmembers()};assert set(entries)==set(members)
   for rel,expected in entries.items():
    f=source.extractfile(members[rel]);assert f and hashlib.sha256(f.read()).hexdigest()==expected
archive_receipt={'status':'uploaded_exact_official_portal_PDF_response','archive_sha256':digest,'archive_bytes':size,'entries':entries,'nas_artifact':evidence,'nas_path':nas_path,'remote_hash_verified':True,'scoped_restore_verified':True}
(card/'source_archive_receipt.json').write_text(json.dumps(archive_receipt,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','source_archive_receipt':'source_archive_receipt.json','source_NAS_sha256':digest,'source_NAS_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');state.write_text(json.dumps({'status':'completed_transport_visual_pending' if valid else 'completed_transport_failed','plan_sha256':sha(card/'plan.json'),'GETs_completed':1,'NAS_scoped_restore_verified':True},indent=2)+'\n')
print(json.dumps({'status':result['status'],'receipt':receipt,'archive_sha256':digest},ensure_ascii=False))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{"status": "PDF_bytes_valid_visual_semantics_pending", "receipt": {"source_url": "https://www.korea.kr/common/download.do?fileId=198574612&tblKey=GMN", "started_at_utc": "2026-10-23T00:37:35.301350+00:00", "completed_at_utc": "2026-10-23T00:37:39.791876+00:00", "curl_exit": 0, "http_status": "200", "effective_url": "https://www.korea.kr/common/download.do?fileId=198574612&tblKey=GMN", "error": null, "body_path": "data/snapshots/kcs_early20_official_mirror_pdf_20261023/attachment.body", "body_bytes": 460569, "body_sha256": "3c25f7dc79c44df47c3e0d64a9639d9c7f9236ff322892e2d480cd445fe4509d", "headers_path": "data/snapshots/kcs_early20_official_mirror_pdf_20261023/attachment.headers", "headers_bytes": 523, "headers_sha256": "465738afccffc2e551ddf2516a6f6a5ca899eca62b72d0b87ba7f7a7eca8bbee"}, "archive_sha256": "430420d84879259319a190973712efd8c15d06c366e2fb56f151eda12a88d352"}


# 韩国海关10月前20日：官方门户PDF原件

这次单独冻结的传输修复只抓取政府政策门户所列的一个 PDF 地址，**HTTP 200、460,569 字节**，SHA-256 `3c25f7dc79c44df47c3e0d64a9639d9c7f9236ff322892e2d480cd445fe4509d`；实际接收始于 **2026-10-23T00:37:35.301350+00:00**。原始响应及头部 NAS 归档 SHA-256 `430420d84879259319a190973712efd8c15d06c366e2fb56f151eda12a88d352`，远端哈希与限定恢复逐对象核对通过。

四页均已渲染审阅。第1页为韩国海关新闻稿，印有 2026 年 10 月 21 日 **09:00 韩国时间即刻分发**及 10 月 1–20 日暂定统计；第4页出口表的半导体行明确写 **27,516 百万美元、同比 +222.5%**。第1页与之前捕获的官方门户 HTML 显示 **275 亿美元**，是 27,516 百万美元换算到亿后四舍五入的数值。第3页提示暂定统计可能在 2027 年 2 月年度确报前修订。细节见 `visual_semantics_review.json`。

本卡证明 **10 月 23 日收到的官方门户 PDF 与 HTML 数值语义一致**，并把 10 月前20日已公开的前置数值保存到 11 月正式月报之前。PDF 上的分发时刻与创建元数据不能单独证明我们收到的是 10 月 21 日当时的原始字节；历史月度首次公开版本仍未合格。**0 SOXX 结果读取、0 拟合、0 账户、0 策略前向日。** 后续若与11月原始正式月报比较，须预先冻结两次来源的同单位和修订处理；不得把本源检查当作 SOXX 胜出。
